In [1]:
import os
import json
from glob import glob
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [2]:
import os
import json
import glob
import random
from typing import Any, Dict, List, Union
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv
from underthesea import word_tokenize

import asyncio
from typing import List, Dict, Any, Awaitable

from pydantic import BaseModel
from openai import AsyncOpenAI
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    Retrying,
    retry_if_exception_type,
    before_sleep_log,
    after_log,
)
import logging
import aiofiles

# Configure logging for the retry mechanism
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [4]:
from openai import AsyncOpenAI, OpenAI

aclient = AsyncOpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY,
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [3]:
# Define the type alias for clarity
JsonData = Union[Dict[str, Any], List[Any]]


def write_json(filepath: str, data: JsonData) -> bool:
    """
    Writes data to a specified JSON file path.

    Args:
        filepath: The path to the output JSON file.
        data: The Python data structure (dict or list) to save.

    Returns:
        True if the write was successful, False otherwise.
    """
    try:
        # Open file in write mode ('w'), specifying UTF-8 encoding
        with open(filepath, "w", encoding="utf-8") as f:
            # Use json.dump for writing. indent=4 makes the file human-readable.
            json.dump(data, f, indent=4)
        print(f"Successfully wrote data to {filepath}")
        return True
    except IOError as e:
        print(f"Error writing to file {filepath}: {e}")
        return False
    except TypeError as e:
        print(
            f"Data type error during serialization: {e}. Check if data is JSON serializable."
        )
        return False


def read_json(filepath: str) -> Union[JsonData, None]:
    """
    Reads and parses data from a specified JSON file path.

    Args:
        filepath: The path to the input JSON file.

    Returns:
        The Python data structure (dict or list) loaded from the file,
        or None if an error occurred.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None

    try:
        # Open file in read mode ('r'), specifying UTF-8 encoding
        with open(filepath, "r", encoding="utf-8") as f:
            # Use json.load to parse the JSON content
            data = json.load(f)
            print(f"Successfully read data from {filepath}")
            return data
    except json.JSONDecodeError as e:
        print(
            f"Error decoding JSON from {filepath}. File might be empty or corrupted: {e}"
        )
        return None
    except IOError as e:
        print(f"Error reading file {filepath}: {e}")
        return None

In [4]:
def init_login():
    load_dotenv()
    login(os.getenv("HUGGINGFACE_TOKEN"))
    print("Login successful")

In [5]:
import os
import httpx
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux i686; rv:109.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/121.0",
]


def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.
    """
    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.google.com/",
            "Connection": "keep-alive",
        }

        with httpx.Client(follow_redirects=True) as client:
            response = client.get(image_url, headers=headers, timeout=timeout)
            response.raise_for_status()  # raise error if 403/404 etc.

            # Create directory if needed
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            with open(save_path, "wb") as f:
                f.write(response.content)

        return True

    except httpx.HTTPStatusError as e:
        print(f"❌ HTTP error {e.response.status_code} for {image_url}")
    except Exception as e:
        print(f"❌ Failed to download {image_url}: {e}")

    return False


def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False


def chunk_list(lst, n):
    """Split a list into n nearly equal chunks."""
    if n <= 0:
        raise ValueError("n must be a positive integer")
    k, m = divmod(len(lst), n)
    return [lst[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n)]

In [7]:
import glob

# datafile = str(DATA_DIR / "german" / "politicians")
datafile = str(DATA_DIR / "raw" / "08122025")
datapaths = glob.glob(datafile + "/wiki**.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_003.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_001.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_002.json']

In [8]:
politician_profiles = []

for path in datapaths:
    json_data = read_json(file_path=path)
    politician_profiles += json_data


len(politician_profiles)

11465

In [6]:
def _get_name_from_url(image_url: str):
    image_name = image_url.split("/")[-1]
    name = image_name.split("-")[:-1]
    return " ".join(name).strip().title()


_get_name_from_url(
    image_url="https://www.thefamouspeople.com/profiles/thumbs/annette-lu-1.jpg"
)

'Annette Lu'

In [9]:
politician_profiles[:10]

[{'title': 'all',
  'url': 'https://en.wikipedia.org/wiki/Lists_of_Philippine_actors',
  'image_url': '',
  'summary': 'Lists of Philippine actors  cover actors from the  Philippines . The lists are organized by age and gender.'},
 {'title': 'Konparu Zenpō',
  'url': 'https://en.wikipedia.org/wiki/Konparu_Zenp%C5%8D',
  'image_url': '',
  'summary': 'Konparu Zenpō  ( 金春 禅鳳 ; 1454–1520? [ 1 ] [ 2 ] )  was a Japanese  Noh  actor and playwright of the Konparu school. He was the grandson of  Konparu Zenchiku . Zenpō\'s plays were more popular and dramatic, novel and crowd-pleasing with large casts and more elaborate effects and sets, than the plays of his grandfather\'s, or his great-grandfather  Zeami \'s, although he did have an appreciation of  yugen  and  wabi  (Zenpō was a pupil of  Shuko  and quoted him as saying "The moon not glimpsed through rifts in clouds holds no interest" [ 3 ] ).'},
 {'title': 'BBC One',
  'url': 'https://en.wikipedia.org/wiki/BBC_One',
  'image_url': 'https:/

In [10]:
final_profiles = []

exist_image_url = []  # Assume the website only use one image for each profile

for filepath in datapaths:
    print(f"Extracting on {filepath}")
    profiles = read_json(file_path=filepath)

    for profile in profiles:
        data = {}
        image_url = profile.get("image_url", "")
        if not image_url:
            # print(f"Image URL is missing for profile: {profile}")
            continue

        if image_url in exist_image_url:
            # print(f"Image URL is duplicated for profile: {profile}")
            continue

        exist_image_url.append(image_url)

        profile_name = profile.get("title", "")
        if not profile_name:
            profile_name = _get_name_from_url(image_url=image_url)
        profile_name = profile_name.title()

        data["name"] = profile_name
        data["mainType"] = profile.get("mainType", "")
        data["dateOfBirth"] = profile.get("dateOfBirth", "")
        data["homePlace"] = profile.get("homePlace", "")
        data["workPlace"] = profile.get("workPlace", "")

        # description = '\n\n'.join([profile.get('about', ''), profile.get('beforeFame', ''), profile.get('trivia', '')])
        # description = profile.get("about", "")
        description = profile.get("summary", "")
        gender = profile.get("gender", "")

        if not gender:
            if "He" in description:
                gender = "Male"
            elif "She" in description:
                gender = "Female"
            else:
                gender = ""

        data["gender"] = gender

        data["description"] = description

        data["image_url"] = image_url
        data["profile_url"] = profile.get("url", "")

        final_profiles.append(data)

len(final_profiles)

Extracting on /home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_003.json
Extracting on /home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_001.json
Extracting on /home/octoopt/workspace/projects/personal/data_enrichment/data/raw/08122025/wiki_actors_002.json


6857

In [11]:
final_profiles[0]

{'name': 'Bbc One',
 'mainType': '',
 'dateOfBirth': '',
 'homePlace': '',
 'workPlace': '',
 'gender': '',
 'description': "BBC One  is a British  free-to-air   public broadcast   television channel  owned and operated by the  BBC . It is the corporation's oldest and  flagship  channel, and is known for broadcasting mainstream programming, which includes  BBC News  television bulletins, primetime drama and entertainment, and live  BBC Sport  events.",
 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/8b/BBC_One_logo_2021.svg/250px-BBC_One_logo_2021.svg.png',
 'profile_url': 'https://en.wikipedia.org/wiki/BBC_One'}

In [12]:
dest_datafile = DATA_DIR / "processed" / "08122025"
dest_datafile.mkdir(exist_ok=True)
write_json(
    data=final_profiles,
    # data=merged_profiles,
    file_path=f"{str(dest_datafile)}/08122025_mrgdprf_bf_extra_001.json",
)

✓ Written JSON to: /home/octoopt/workspace/projects/personal/data_enrichment/data/processed/08122025/08122025_mrgdprf_bf_extra_001.json


True

In [13]:
final_profiles[0]

{'name': 'Bbc One',
 'mainType': '',
 'dateOfBirth': '',
 'homePlace': '',
 'workPlace': '',
 'gender': '',
 'description': "BBC One  is a British  free-to-air   public broadcast   television channel  owned and operated by the  BBC . It is the corporation's oldest and  flagship  channel, and is known for broadcasting mainstream programming, which includes  BBC News  television bulletins, primetime drama and entertainment, and live  BBC Sport  events.",
 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/8b/BBC_One_logo_2021.svg/250px-BBC_One_logo_2021.svg.png',
 'profile_url': 'https://en.wikipedia.org/wiki/BBC_One'}

In [14]:
import random

# Create a copy of final_profiles and shuffle it
rand_profiles = final_profiles.copy()
random.shuffle(rand_profiles)

rand_profiles[0]

{'name': 'Shouma Kai',
 'mainType': '',
 'dateOfBirth': '',
 'homePlace': '',
 'workPlace': '',
 'gender': 'Male',
 'description': 'Shouma Kai  ( 甲斐 翔真 ,  Kai Shōma ; born 14 November 1997, in  Tokyo ) [ 1 ]  is a Japanese actor. He is represented by  Amuse, Inc.',
 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Kai_Shoma_from_%22Shashin_Koshien_Summer_in_0.5_Seconds%22_at_Opening_Ceremony_of_the_Tokyo_International_Film_Festival_2017_%2839491506724%29.jpg/250px-Kai_Shoma_from_%22Shashin_Koshien_Summer_in_0.5_Seconds%22_at_Opening_Ceremony_of_the_Tokyo_International_Film_Festival_2017_%2839491506724%29.jpg',
 'profile_url': 'https://en.wikipedia.org/wiki/Shouma_Kai'}

In [15]:
dest_datafile = DATA_DIR / "processed" / "08122025"

chunked_profiles = chunk_list(lst=rand_profiles, n=2)


for idx in range(len(chunked_profiles)):
    placeholder = 3 + idx
    profiles = chunked_profiles[idx]
    write_json(
        data=profiles,
        # data=merged_profiles,
        file_path=f"{str(dest_datafile)}/mrgdprf_bf_{str(placeholder)}.json",
    )

✓ Written JSON to: /home/octoopt/workspace/projects/personal/data_enrichment/data/processed/08122025/mrgdprf_bf_3.json
✓ Written JSON to: /home/octoopt/workspace/projects/personal/data_enrichment/data/processed/08122025/mrgdprf_bf_4.json


In [19]:
final_profiles = read_json(file_path=f"{str(dest_datafile)}/mrgdprf_bf_4.json")

len(final_profiles)

3428

In [20]:
# """
# Only get profile that get image
# """

# usable_profiles = []
# save_dir = str(DATA_DIR / "images" / "mrgdprf_bf_0")


# for idx in tqdm(range(len(final_profiles))):
#     try:
#         profile = final_profiles[idx]
#         image_url = profile["image_url"]
#         img_name = profile["name"].replace(" ", "_")
#         local_path = f"{save_dir}/{img_name}.jpg"
#         profile["local_path"] = local_path
#         is_success = download_image(image_url, local_path)
#         if is_success:
#             usable_profiles.append(profile)

#     except Exception as e:
#         print(e)
#         continue


from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm
import os

# ---------- CONFIG ----------
save_dir = Path(DATA_DIR) / "images" / "08122025" / "mrgdprf_4"
save_dir.mkdir(parents=True, exist_ok=True)
usable_profiles = []


# ---------- FUNCTION ----------
def process_profile(profile):
    """Download image for a single profile."""
    try:
        image_url = profile["image_url"]
        img_name = profile["name"].replace(" ", "_")
        local_path = save_dir / f"{img_name}.jpg"
        profile["local_path"] = str(local_path)
        is_success = download_image(image_url, local_path)
        return profile if is_success else None
    except Exception as e:
        print(f"Error: {e}")
        return None


# ---------- MULTI-THREAD EXECUTION ----------
max_workers = os.cpu_count() or 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_profile, p) for p in final_profiles]

    for future in tqdm(
        as_completed(futures), total=len(futures), desc="Downloading images"
    ):
        result = future.result()
        if result:
            usable_profiles.append(result)

print(f"✅ Collected {len(usable_profiles)} usable profiles.")

2025-12-08 23:21:12,348 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Val_Avery_in_Bonanza_%28Breed_of_Violence%29.jpg/250px-Val_Avery_in_Bonanza_%28Breed_of_Violence%29.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:12,367 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/4/44/2AM_in_Manila_from_Apr_3.jpg/250px-2AM_in_Manila_from_Apr_3.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:12,381 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/8/81/Flag_of_the_Dutch_East_India_Company.svg/250px-Flag_of_the_Dutch_East_India_Company.svg.png "HTTP/1.1 200 OK"
2025-12-08 23:21:12,390 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Babu_Antony_in_Karinkunnam_6s_movie.jpg/250px-Babu_Antony_in_Karinkunnam_6s_movie.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:12,394 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/1/17/Mahmoud_Samir_Fayed_the_creator_of_PWCT.jpg

❌ HTTP error 404 for https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/Komal_Jha_at_Event.jpg/250px-Komal_Jha_at_Event.jpg


2025-12-08 23:21:53,475 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/Park_Sun-young_%28South_Korean_actress%2C_born_August_21%2C_1976%29.jpg/250px-Park_Sun-young_%28South_Korean_actress%2C_born_August_21%2C_1976%29.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:53,507 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/9/90/Vineet_Kumar_2022.jpg/250px-Vineet_Kumar_2022.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:53,508 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/5/5a/Elahe_Hesari%2C_Highlight_movie_press_conference%2C_36th_Fajr_Film_Festival_%2834264%29.jpg/250px-Elahe_Hesari%2C_Highlight_movie_press_conference%2C_36th_Fajr_Film_Festival_%2834264%29.jpg "HTTP/1.1 200 OK"
2025-12-08 23:21:53,584 - INFO - HTTP Request: GET https://upload.wikimedia.org/wikipedia/commons/thumb/9/9d/DimpleKapadiaPichvai_%28cropped%29.png/250px-DimpleKapadiaPichvai_%28cropped%29.png "HTTP/1.1 200 OK"
2025-12-08 23:21:5

✅ Collected 3427 usable profiles.


In [21]:
write_json(
    data=usable_profiles,
    file_path=f"{str(dest_datafile)}/08122025_mrgdprf_aft_4.json",
)

✓ Written JSON to: /home/octoopt/workspace/projects/personal/data_enrichment/data/processed/08122025/08122025_mrgdprf_aft_4.json


True